# Notebook for configuring jobs and processing
#### Fist we export the needed

In [15]:
import numpy as np
import pickle, sys, os, json, glob, tables, subprocess
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from astropy.coordinates import SkyCoord
from astropy import units as u

import lstchain

sys.path.insert(0, os.path.join(os.getcwd(), "../scripts/"))
import utils

# --- Version --- #
root_lstchain = lstchain.__path__[0]
version_lstchain = f"v{(lstchain.__version__).split('.dev')[0]}"
print(f"Using lstchain version {version_lstchain} from:\n{root_lstchain}")

# STANDARD paths ---------
root_dl1 = "/fefs/aswg/data/real/DL1/????????/v*/tailcut*/"
root_dl2 = "/fefs/aswg/data/real/DL2/????????/v*/tailcut*/nsb_tuning_*/"

default_config_lstchain_dl2 = os.path.join(root_lstchain, "data/lstchain_standard_config.json")
default_config_lstchain_dl3 = os.path.join(root_lstchain, "../docs/examples/irf_dl3_tool_config.json")

root_rfs = "/fefs/aswg/data/models/AllSky/20240918_v0.10.12_allsky_nsb_tuning_*/"
root_mcs = "/fefs/aswg/data/mc/DL2/AllSky/20240918_v0.10.12_allsky_nsb_tuning_*/TestingDataset/"

Using lstchain version v0.10.18 from:
/fefs/aswg/workspace/juan.jimenez/softs/analysis/cta-lstchain/lstchain


In [16]:
source_name = "S241125n"
source_coords = SkyCoord(ra=58.079, dec=69.689, unit=u.deg)
str_dec = "dec_6676"

gh_dyn_cut = 90

# Some options
overwrite      = True
compute_irfs   = True
process_inline = True
interpolate_irf = True

In [17]:
# Root path of this script
root = os.getcwd() + "/"
# Path to store the configuration file we are going to use
root_config = root + "config/"

# Config files
file_config_job = os.path.join(root_config, "config_jobs_runs.txt")

file_config_lstchain_dl2 = os.path.join(root_config, "config_lstchain_dl2.json")
file_config_lstchain_dl3 = os.path.join(root_config, "config_lstchain_dl3.json")

root_data = os.path.join(
    "/fefs/aswg/workspace/juan.jimenez/data/", "real", "mono",
    f"{source_name}", version_lstchain, "GammaDiffuse", "prod_standard"
)
root_data_s = os.path.join(
    "/fefs/aswg/workspace/juan.jimenez/data/", "real", "mono",
    f"{source_name}", version_lstchain, "GammaDiffuse", "prod_light_scaling"
)

str_gh = f"gh_dyn{gh_dyn_cut}"

dir_irf = os.path.join(
    "/fefs/aswg/workspace/juan.jimenez/data/", "mc", "mono", "IRF", version_lstchain, 
    f"NSB*", "GammaDiffuse", str_dec, str_gh
) # ???????????????????????? CHANGE DIFFUSE

dir_dl1_s = os.path.join(root_data_s, "DL1")
dir_dl2_s = os.path.join(root_data_s, "DL2")
dir_dl3_s = os.path.join(root_data_s, "DL3", str_gh)
dir_dl3   = os.path.join(root_data, "DL3", str_gh)

In [18]:
# Create the paths that do not exist
for path in [dir_dl2_s, dir_dl3, dir_dl3_s]:
    os.makedirs(path, exist_ok=True)

### Configuration

In [19]:
config_changes_dl2 = {
    "events_filters": {"intensity": [50, float("inf")],},
}

config_changes_dl3 = {
    "EventSelector": {"filters": {"intensity": [50, float("inf")],},},
    "DataBinning": {"fov_offset_min": 0.0, "fov_offset_max": 2.5, "fov_offset_n_edges": 6,},
    "DL3Cuts": {
        "gh_efficiency": gh_dyn_cut / 100,
    }
}

In [20]:
with open(default_config_lstchain_dl3, "r") as file_dl3, open(default_config_lstchain_dl2, "r") as file_dl2:
    standard_config_dl3 = json.load(file_dl3)
    standard_config_dl2 = json.load(file_dl2)

dict_config_dl2 = utils.modify_json_data(standard_config_dl2, config_changes_dl2)
dict_config_dl3 = utils.modify_json_data(standard_config_dl3, config_changes_dl3)

utils.write_json_file(dict_config_dl2, file_config_lstchain_dl2)
utils.write_json_file(dict_config_dl3, file_config_lstchain_dl3)

##### Getting the run numbers

In [21]:
# Reading the config file
jobs_list = np.atleast_1d(np.loadtxt(file_config_job, dtype="str"))

obs_ids = np.sort(np.unique([int(j.split("_")[0]) for j in jobs_list]))

# <span style="color:blue">1. DL1 to DL2</span>

In [23]:
%%time
# Iterating over all LST runs
for obs_id in obs_ids[:]:
    i = list(obs_ids).index(obs_id)
    print(f"\nRunning DL1 --> DL2 for LST Run {obs_id} {i}/{len(obs_ids)}...\n")

    # Finding the run file in `data_root`
    file_dl1_s = glob.glob(os.path.join(dir_dl1_s, f"dl1_LST-1.Run{obs_id:05}.h5"))[0]
    
    # Finding the RF folder
    # First we find the DL2 file, in order to get the NSB level
    file_dl2 = glob.glob(os.path.join(root_dl2, f"dl2_LST-1.Run{obs_id:05}.h5"))[0]
    str_nsb_tuning = file_dl2.split("nsb_tuning_")[1].split("/")[0]
    
    dir_rf = os.path.join(root_rfs.replace("nsb_tuning_*", f"nsb_tuning_{str_nsb_tuning}"), str_dec)
    
    command_dl1dl2 = f"lstchain_dl1_to_dl2 --input-files {file_dl1_s} --path-models {dir_rf} "
    command_dl1dl2 = command_dl1dl2 + f"--output-dir {dir_dl2_s} --config {file_config_lstchain_dl2}"
    
    str_output = f"-o ./output/slurm_output/dl1_to_dl2_light_scaling_{obs_id}.out"
    slurm_command = f"sbatch -p short --mem=80000 -J dl1_to_dl2_joint {str_output} --wrap='{command_dl1dl2}'"

    command = command_dl1dl2 if process_inline else slurm_command
    subprocess.run(command, shell=True, text=True)


Running DL1 --> DL2 for LST Run 19799 0/13...



2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): Data contains not-predictable events.
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): Column | Number of non finite values
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): width : 1618304
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): time_gradient : 1803292
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): wl : 1618309
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): x : 1618304
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): length : 1618304
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): y : 1618304
2025-03-13 17:36:11,897 WARNING [lstchain.reco.utils] (utils.filter_events): leakage_intensity_width_2 : 1618304
2025-03-13 17:36:11,898 WARNING [lstchain.reco.utils] (utils.filter_events): skewness : 1618309
2025-0


Running DL1 --> DL2 for LST Run 19800 1/13...



2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): Data contains not-predictable events.
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): Column | Number of non finite values
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): wl : 416211
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): x : 416209
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): width : 416209
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): leakage_intensity_width_2 : 416209
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): y : 416209
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): length : 416209
2025-03-13 17:38:56,341 WARNING [lstchain.reco.utils] (utils.filter_events): log_intensity : 416209
2025-03-13 17:38:56,342 WARNING [lstchain.reco.utils] (utils.filter_events): kurtosis : 416211
2025-03-13 17:


Running DL1 --> DL2 for LST Run 19801 2/13...



...


Running DL1 --> DL2 for LST Run 19810 11/13...



2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): Data contains not-predictable events.
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): Column | Number of non finite values
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): log_intensity : 492347
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): x : 492347
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): kurtosis : 492347
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): leakage_intensity_width_2 : 492347
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): width : 492347
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): y : 492347
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): time_gradient : 502570
2025-03-13 18:20:18,436 WARNING [lstchain.reco.utils] (utils.filter_events): skewness : 492347
2


Running DL1 --> DL2 for LST Run 19811 12/13...



2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): Data contains not-predictable events.
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): Column | Number of non finite values
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): x : 154641
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): wl : 154641
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): y : 154641
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): kurtosis : 154641
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): length : 154641
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): width : 154641
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): time_gradient : 157307
2025-03-13 18:22:43,684 WARNING [lstchain.reco.utils] (utils.filter_events): skewness : 154641
2025-03-13 18:22:43,684 WARNING

CPU times: user 1.25 s, sys: 3.28 s, total: 4.54 s
Wall time: 50min 32s


In [24]:
!squeue -u juan.jimenez
# !scancel -u juan.jimenez 

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON) 
          44271846     short dl1_to_d juan.jim CG       0:00      1 cp44 
          44273177     short create_r juan.jim  R      20:26      1 cp19 
          44273174     short create_r juan.jim  R      20:30      1 cp19 
          44273175     short create_r juan.jim  R      20:30      1 cp19 


# <span style="color:blue">2. Generating the IRFs</span>
##### First we gwt the NSB levels

In [25]:
nsb_levels, dict_nsb_levels, dict_nsbwise = [], {}, {}
for obs_id in obs_ids:
    query_dl2 = glob.glob(root_dl2 + f"dl2_LST-1.Run{obs_id}.h5")

    str_nsb = query_dl2[0].split("nsb_tuning_")[1].split("/")[0]
    nsb_levels.append(str_nsb)
    dict_nsb_levels[obs_id] = str_nsb
    
    dict_nsbwise[str_nsb] = {
        "dir_irf" : dir_irf.replace("NSB*", f"NSB{str_nsb}"),
        "dir_mc" : os.path.join(
            root_mcs.replace("nsb_tuning_*", f"nsb_tuning_{str_nsb}"), "GammaDiffuse", str_dec
        ),
    }
    
print(f"A total of {len(np.unique(nsb_levels))} NSB levels needed: {np.unique(nsb_levels)}")

A total of 3 NSB levels needed: ['0.07' '0.14' '0.22']


##### Then generating the IRFs

In [23]:
%%time
if compute_irfs:
    for nsb in dict_nsbwise.keys():
        query_mc_dl2 = np.sort(glob.glob(os.path.join(dict_nsbwise[nsb]["dir_mc"], "*", "*.h5")))
        print(f"\nComputing IRFs for NSB {nsb}... ({len(query_mc_dl2)} files)")

        for j, file_mc in enumerate(query_mc_dl2):
            
            print(f"\nComputing IRF of {file_mc}, NSB{nsb}: {j}/{len(query_mc_dl2)}")

            # We define a filename for each IRF with all MC information
            file_irf = os.path.join(
                dict_nsbwise[nsb]["dir_irf"], 
                file_mc.split("/")[-1].replace("dl2_", "irf_").replace(".h5", ".fits.gz")
            )
            os.makedirs(os.path.dirname(file_irf), exist_ok=True)

            # Then we build the command
            str_args  = f"--input-gamma-dl2={file_mc} --output-irf-file={file_irf}"
            str_args += f" --config={file_config_lstchain_dl3} --point-like"

            # With some extra options
            str_args += " --overwrite" if overwrite else ""
            print(f"Computing point-like IRF...")

            command = f"lstchain_create_irf_files {str_args}"
            subprocess.run(command, shell=True, text=True)


Computing IRFs for NSB 0.22... (20 files)

Computing IRF of /fefs/aswg/data/mc/DL2/AllSky/20240918_v0.10.12_allsky_nsb_tuning_0.22/TestingDataset/GammaDiffuse/dec_6676/node_corsika_theta_38.108_az_3.026_/dl2_20240918_v0.10.12_allsky_nsb_tuning_0.22_GammaDiffuse_test_dec_6676_node_corsika_theta_38.108_az_3.026__merged.h5, NSB0.22: 0/20
Computing point-like IRF...


2025-03-13 15:50:07,601 WARNING [lstchain.IRFFITSWriter] (lstchain_create_irf_files.setup): Overwriting /fefs/aswg/workspace/juan.jimenez/data/mc/mono/IRF/v0.10.18/NSB0.22/GammaDiffuse/dec_6676/gh_dyn90/irf_20240918_v0.10.12_allsky_nsb_tuning_0.22_GammaDiffuse_test_dec_6676_node_corsika_theta_38.108_az_3.026__merged.fits.gz



Computing IRFs for NSB 0.14... (20 files)

Computing IRF of /fefs/aswg/data/mc/DL2/AllSky/20240918_v0.10.12_allsky_nsb_tuning_0.14/TestingDataset/GammaDiffuse/dec_6676/node_corsika_theta_38.108_az_3.026_/dl2_20240918_v0.10.12_allsky_nsb_tuning_0.14_GammaDiffuse_test_dec_6676_node_corsika_theta_38.108_az_3.026__merged.h5, NSB0.14: 0/20
Computing point-like IRF...

Computing IRFs for NSB 0.07... (20 files)

Computing IRF of /fefs/aswg/data/mc/DL2/AllSky/20240918_v0.10.12_allsky_nsb_tuning_0.07/TestingDataset/GammaDiffuse/dec_6676/node_corsika_theta_38.108_az_3.026_/dl2_20240918_v0.10.12_allsky_nsb_tuning_0.07_GammaDiffuse_test_dec_6676_node_corsika_theta_38.108_az_3.026__merged.h5, NSB0.07: 0/20
Computing point-like IRF...
CPU times: user 9.51 ms, sys: 14.4 ms, total: 23.9 ms
Wall time: 34.7 s


# <span style="color:blue">3. DL2 to DL3 for standard and for scaled data</span>
### <span style="color:blue">3.1 For scaled data</span>

In [26]:
%%time
for obs_id in obs_ids[:]:
    i = list(obs_ids).index(obs_id)
    print(f"\nRunning DL2 --> DL3 for LST Run {obs_id}...\n")    
    nsb = nsb_levels[i]
    
    # Finding the run file in `data_root`
    file_dl2 = glob.glob(dir_dl2_s + f"/dl2_LST-1.Run{obs_id:05}.h5")[0]
    
    # We build the command that need to be run
    str_args  = f" --input-dl2={file_dl2} --input-irf-path={dict_nsbwise[nsb]['dir_irf']}"
    str_args += f" --output-dl3-path={dir_dl3_s} --config={file_config_lstchain_dl3}"
    str_args += f" --source-name {source_name}"
    str_args += f" --source-ra={source_coords.ra.deg}deg --source-dec={source_coords.dec.deg}deg"
    
    # Some options
    str_args += " --overwrite"  if overwrite else ""
    str_args += " --interp-method cubic" if interpolate_irf else ""
    str_args += " --use-nearest-irf-node" if not interpolate_irf else ""
    
    python_command = f"lstchain_create_dl3_file {str_args}"
    subprocess.run(python_command, shell=True, text=True)
    


Running DL2 --> DL3 for LST Run 19799...



/fefs/aswg/workspace/juan.jimenez/.conda/envs/baseenv/lib/python3.11/site-packages/tables/group.py:1220: UserWarning: problems loading leaf ``/provenance/dl1_to_dl2``::

  variable length strings are not supported yet

The leaf will become an ``UnImplemented`` node.
  warnings.warn(
2025-03-13 19:17:12,046 WARNING [lstchain.high_level.interpolate] (interpolate.interpolate_irf): The interpolation of BACKGROUND is not yet supported


The metadata are comparable
The other parameter axes data are comparable

Running DL2 --> DL3 for LST Run 19800...



/fefs/aswg/workspace/juan.jimenez/.conda/envs/baseenv/lib/python3.11/site-packages/tables/group.py:1220: UserWarning: problems loading leaf ``/provenance/dl1_to_dl2``::

  variable length strings are not supported yet

The leaf will become an ``UnImplemented`` node.
  warnings.warn(
2025-03-13 19:17:30,418 WARNING [lstchain.high_level.interpolate] (interpolate.interpolate_irf): The interpolation of BACKGROUND is not yet supported


The metadata are comparable
The other parameter axes data are comparable

Running DL2 --> DL3 for LST Run 19801...



...

The metadata are comparable
The other parameter axes data are comparable

Running DL2 --> DL3 for LST Run 19810...



/fefs/aswg/workspace/juan.jimenez/.conda/envs/baseenv/lib/python3.11/site-packages/tables/group.py:1220: UserWarning: problems loading leaf ``/provenance/dl1_to_dl2``::

  variable length strings are not supported yet

The leaf will become an ``UnImplemented`` node.
  warnings.warn(
2025-03-13 19:21:29,309 WARNING [lstchain.high_level.interpolate] (interpolate.interpolate_irf): The interpolation of BACKGROUND is not yet supported


The metadata are comparable
The other parameter axes data are comparable

Running DL2 --> DL3 for LST Run 19811...



/fefs/aswg/workspace/juan.jimenez/.conda/envs/baseenv/lib/python3.11/site-packages/tables/group.py:1220: UserWarning: problems loading leaf ``/provenance/dl1_to_dl2``::

  variable length strings are not supported yet

The leaf will become an ``UnImplemented`` node.
  warnings.warn(
2025-03-13 19:21:48,060 WARNING [lstchain.high_level.interpolate] (interpolate.interpolate_irf): The interpolation of BACKGROUND is not yet supported


The metadata are comparable
The other parameter axes data are comparable
CPU times: user 75 ms, sys: 22.8 ms, total: 97.8 ms
Wall time: 5min 9s


### <span style="color:blue">3.2 For standard data</span>

In [ ]:
%%time
for obs_id in obs_ids[:]:
    i = list(obs_ids).index(obs_id)
    print(f"\nRunning DL2 --> DL3 for LST Run {obs_id}...\n")    
    
    # Finding the run file in `data_root`
    query_file_dl2 = glob.glob(root_dl2 + f"dl2_LST-1.Run{obs_id:05}.h5")
    
    # We build the command that need to be run
    str_args  = f" --input-dl2={file_dl2} --input-irf-path={dir_irf[i]}"
    str_args += f" --output-dl3-path={dir_dl3} --config={file_config_lstchain_dl3} --source-name {source_name}"
    str_args += f" --source-ra={source_coords.ra.deg}deg --source-dec={source_coords.dec.deg}deg"
    
    # Some options
    str_args += " --overwrite"  if overwrite else ""
    str_args += " --interp-method cubic" if interpolate_irf else ""
    str_args += " --use-nearest-irf-node" if not interpolate_irf else ""
    
    python_command = f"lstchain_create_dl3_file {str_args}"
    subprocess.run(python_command, shell=True, text=True) 

# <span style="color:blue">4. Create index files</span>

In [27]:
str_args  = f"--input-dl3-dir={dir_dl3_s} --file-pattern=dl3*fits"
str_args += " --overwrite"  if overwrite else ""

slurm_command = f"lstchain_create_dl3_index_files {str_args}"
subprocess.run(slurm_command, shell=True, text=True);

In [ ]:
str_args  = f"--input-dl3-dir={dir_dl3} --file-pattern=dl3*fits"
str_args += " --overwrite"  if overwrite else ""

slurm_command = f"lstchain_create_dl3_index_files {str_args}"
subprocess.run(slurm_command, shell=True, text=True);